# **Universal YOLOv8 Training (ASL)**
### Works on: Google Colab & Kaggle Kernels
### Supports: Google Drive (Colab only), Manual Upload, & Roboflow

In [ ]:
# Cell 1: Environment Setup & GPU Check
import os
import sys
import torch

# Detect Environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = 'kaggle_web_client' in os.environ

if IS_COLAB:
    ROOT_DIR = '/content'
    print("✅ Detected Environment: Google Colab")
    # Mount Drive only if in Colab
    from google.colab import drive
    drive.mount('/content/drive')
elif IS_KAGGLE:
    ROOT_DIR = '/kaggle/working'
    print("✅ Detected Environment: Kaggle Kernel")
else:
    ROOT_DIR = '.'
    print("✅ Detected Environment: Local/Other")

# Install Ultralytics (YOLOv8)
!pip install ultralytics -q
import ultralytics
from ultralytics import YOLO
from IPython.display import clear_output

clear_output()
ultralytics.checks()
print(f"\n🔥 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")

---

In [ ]:
# Cell 2: Data Loading (Choose ONE Method)

# === METHOD A: Roboflow (Easiest) ===
# Uncomment and fill your API Key
# !pip install roboflow -q
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("workspace-name").project("project-name")
# dataset = project.version(1).download("yolov8")
# dataset_location = dataset.location

# === METHOD B: Manual Zip / Drive (Universal) ===
# If using Kaggle: Upload dataset via 'Add Data' button or upload zip.
# If using Colab: Path is usually /content/drive/MyDrive/...

zip_path = "" # <--- PASTE PATH TO ZIP HERE (e.g., /content/drive/MyDrive/ASL.zip)
dataset_location = os.path.join(ROOT_DIR, "dataset_asl")

if zip_path and os.path.exists(zip_path):
    import shutil
    if os.path.exists(dataset_location):
        shutil.rmtree(dataset_location)
    os.makedirs(dataset_location, exist_ok=True)
    print("Unzipping dataset...")
    !unzip -q "{zip_path}" -d "{dataset_location}"
    print(f"✅ Dataset extracted to {dataset_location}")
else:
    # Fallback: Check if user already extracted it or used Roboflow above
    if 'dataset_location' not in locals():
        dataset_location = os.path.join(ROOT_DIR, "dataset_asl") # Default fallback
        print(f"⚠️ No zip path provided. Assuming dataset is already at {dataset_location}")

In [ ]:
# Cell 3: Fix data.yaml Paths (Critical for Kaggle/Colab compatibility)
import yaml
import glob

# Find data.yaml automatically
yaml_files = glob.glob(f"{dataset_location}/**/data.yaml", recursive=True)

if yaml_files:
    target_yaml = yaml_files[0]
    print(f"Found config at: {target_yaml}")
    
    with open(target_yaml, 'r') as f:
        data = yaml.safe_load(f)
    
    # Force Absolute Paths
    # Note: We assume 'train' and 'valid' folders are in the same dir as data.yaml
    base_path = os.path.dirname(target_yaml)
    data['path'] = base_path
    data['train'] = 'train/images'
    data['val'] = 'valid/images'
    if 'test' in data: data['test'] = 'test/images'
    
    with open(target_yaml, 'w') as f:
        yaml.dump(data, f)
    print("✅ data.yaml updated with absolute paths.")
else:
    print("❌ data.yaml not found! Check your dataset structure.")

---

In [ ]:
# Cell 4: Train YOLOv8 Small
# We use 'yolov8s.pt' (Small) for better accuracy than Nano, but still fast.

model = YOLO('yolov8s.pt')

results = model.train(
    data=target_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    name='asl_model',
    patience=10,
    project=os.path.join(ROOT_DIR, 'runs/detect'), # Save runs to writable dir
    exist_ok=True
)

In [ ]:
# Cell 5: Visualize Confusion Matrix
from IPython.display import Image, display

cm_path = os.path.join(ROOT_DIR, 'runs/detect/asl_model/confusion_matrix.png')
if os.path.exists(cm_path):
    display(Image(filename=cm_path, width=800))
else:
    print("Confusion matrix not found yet.")

In [ ]:
# Cell 6: Export Model
# If on Colab, this saves to Drive. 
# If on Kaggle, you will find the file in the 'Output' section on the right.

best_weight_path = os.path.join(ROOT_DIR, 'runs/detect/asl_model/weights/best.pt')

if IS_COLAB:
    dest = '/content/drive/MyDrive/ASL_YOLOv8s_Final.pt'
    if os.path.exists(best_weight_path):
        import shutil
        shutil.copy(best_weight_path, dest)
        print(f"✅ Saved to Google Drive: {dest}")
else:
    print(f"✅ Model trained. Download 'best.pt' from: {best_weight_path}")